# Chương 3 – Nâng cao chất lượng ảnh
## Bài tập thực hành

Mục tiêu:
- Thực hành các **toán tử điểm** (âm bản, phân ngưỡng, biến đổi log, biến đổi hàm mũ/gamma).
- Thực hành **histogram** và **cân bằng lược đồ xám** (Histogram Equalization, CLAHE).
- Thực hành các **bộ lọc không gian** (trung bình, làm sắc nét, trung vị) để khử nhiễu / cải thiện ảnh.
- Áp dụng tổng hợp các kỹ thuật trên để cải thiện chất lượng một số ảnh thực tế.

In [ ]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

%matplotlib inline

IMG_DIR = 'images'

## 1. Toán tử điểm (Point operators)

Toán tử điểm có dạng tổng quát `g(x, y) = T[f(x, y)]`, trong đó giá trị điểm ảnh đầu ra chỉ phụ thuộc vào
giá trị điểm ảnh đầu vào tại đúng vị trí đó (không phụ thuộc các điểm lân cận).

### 1.1. Biến đổi âm bản (Negative transform): `s = 255 - r`

**Yêu cầu**: viết hàm `negative_transform(img)` và áp dụng cho ảnh `HoGuom.jpg` (ảnh xám).

In [ ]:
def negative_transform(img):
    result = 255 - img
    return result

img = cv2.imread(os.path.join(IMG_DIR, 'HoGuom.jpg'), cv2.IMREAD_GRAYSCALE)
neg = negative_transform(img)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(img, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Ảnh âm bản'); plt.imshow(neg, cmap='gray')
plt.show()

### 1.2. Phân ngưỡng (Thresholding)

**Yêu cầu**: viết hàm `threshold_transform(img, T)` trả về ảnh nhị phân (0 hoặc 255) theo ngưỡng `T`,
áp dụng cho ảnh `unequalized.jpg`.

In [ ]:
def threshold_transform(img, T):
    result = np.where(img >= T, 255, 0).astype(np.uint8)
    return result

img2 = cv2.imread(os.path.join(IMG_DIR, 'unequalized.jpg'), cv2.IMREAD_GRAYSCALE)
binary = threshold_transform(img2, 128)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(img2, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau phân ngưỡng'); plt.imshow(binary, cmap='gray')
plt.show()

## 2. Biến đổi Log và biến đổi hàm mũ (Gamma correction)

### 2.1. Biến đổi Log: `s = c * log(1 + r)`, với `c = 255 / log(1 + max(r))`

In [ ]:
log_img = cv2.imread(os.path.join(IMG_DIR, 'log_img.jpg'), cv2.IMREAD_GRAYSCALE)
plt.imshow(log_img, cmap='gray')
plt.show()

In [ ]:
def log_transform(img):
    img = img.astype(np.float64)
    maximum = img.max()
    if maximum == 0:
        result = np.zeros_like(img)
    else:
        c = 255.0 / np.log1p(maximum)
        result = np.clip(np.rint(c * np.log1p(img)), 0, 255)
    return np.uint8(result)

log_result = log_transform(log_img)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(log_img, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau biến đổi Log'); plt.imshow(log_result, cmap='gray')
plt.show()

### 2.2. Biến đổi hàm mũ / Gamma correction: `O = (I / 255) ^ gamma * 255`

Với `gamma < 1`: ảnh tối được tăng sáng. Với `gamma > 1`: ảnh được giảm sáng.

**Yêu cầu**: viết hàm `gamma_correction(img, gamma)` và thử với nhiều giá trị gamma khác nhau trên ảnh
`HoGuom.jpg`.

In [ ]:
def gamma_correction(img, gamma):
    normalized = img.astype(np.float64) / 255.0
    result = np.clip(np.rint(255.0 * np.power(normalized, gamma)), 0, 255).astype(np.uint8)
    return result

img3 = cv2.imread(os.path.join(IMG_DIR, 'HoGuom.jpg'), cv2.IMREAD_GRAYSCALE)
gammas = [0.4, 0.6, 1.0, 2.0, 4.0]

plt.figure(figsize=(4 * len(gammas), 4))
for i, g in enumerate(gammas):
    plt.subplot(1, len(gammas), i + 1)
    plt.imshow(gamma_correction(img3, g), cmap='gray')
    plt.title(f'gamma={g}')
    plt.axis('off')
plt.show()

## 3. Histogram và Cân bằng lược đồ xám

### 3.1. Cân bằng histogram toàn ảnh (Histogram Equalization)

**Yêu cầu**: dùng `cv2.equalizeHist` để cân bằng lược đồ xám cho ảnh `unequalized.jpg`, hiển thị ảnh và
histogram trước/sau.

In [ ]:
img4 = cv2.imread(os.path.join(IMG_DIR, 'unequalized.jpg'), cv2.IMREAD_GRAYSCALE)

equ = cv2.equalizeHist(img4)

plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1); plt.title('Ảnh gốc'); plt.imshow(img4, cmap='gray')
plt.subplot(2, 2, 2); plt.title('Sau cân bằng'); plt.imshow(equ, cmap='gray')
plt.subplot(2, 2, 3); plt.hist(img4.ravel(), 256, [0, 256])
plt.subplot(2, 2, 4); plt.hist(equ.ravel(), 256, [0, 256])
plt.show()

### 3.2. Khi cân bằng toàn ảnh không hiệu quả: CLAHE

Với ảnh có vùng sáng/tối phân bố không đều (ví dụ `statue_bright.jpg`), cân bằng histogram toàn ảnh có
thể không cho kết quả tốt. `CLAHE` (Contrast Limited Adaptive Histogram Equalization) chia ảnh thành các
khối nhỏ (tile) và cân bằng riêng từng khối.

**Yêu cầu**: so sánh 3 kết quả: ảnh gốc, sau `cv2.equalizeHist`, và sau `cv2.createCLAHE`.

In [ ]:
statue = cv2.imread(os.path.join(IMG_DIR, 'statue_bright.jpg'), cv2.IMREAD_GRAYSCALE)

statue_equ = cv2.equalizeHist(statue)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
statue_clahe = clahe.apply(statue)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.title('Ảnh gốc'); plt.imshow(statue, cmap='gray')
plt.subplot(1, 3, 2); plt.title('equalizeHist'); plt.imshow(statue_equ, cmap='gray')
plt.subplot(1, 3, 3); plt.title('CLAHE'); plt.imshow(statue_clahe, cmap='gray')
plt.show()

## 4. Lọc không gian (Spatial filtering)

### 4.1. Bộ lọc trung bình (Averaging / smoothing filter)

**Yêu cầu**: tạo kernel trung bình 5x5 (mỗi phần tử = 1/25) bằng `numpy` và áp dụng bằng `cv2.filter2D`
lên ảnh màu `himym.jpg`.

In [ ]:
img5 = cv2.imread(os.path.join(IMG_DIR, 'himym.jpg'))
img5 = cv2.cvtColor(img5, cv2.COLOR_BGR2RGB)

kernel = np.ones((5, 5), dtype=np.float32) / 25.0
smoothed = cv2.filter2D(img5, -1, kernel)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(img5)
plt.subplot(1, 2, 2); plt.title('Sau lọc trung bình 5x5'); plt.imshow(smoothed)
plt.show()

### 4.2. Bộ lọc làm sắc nét (Sharpening filter)

**Yêu cầu**: dùng kernel sharpening `[[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]` với `cv2.filter2D`.

In [ ]:
kernel_sharpening = np.array([[-1, -1, -1],
                              [-1,  9, -1],
                              [-1, -1, -1]], dtype=np.float32)
sharpened = cv2.filter2D(img5, -1, kernel_sharpening)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(img5)
plt.subplot(1, 2, 2); plt.title('Sau làm sắc nét'); plt.imshow(sharpened)
plt.show()

## 5. Khôi phục ảnh: khử nhiễu bằng bộ lọc không gian

### 5.1. Nhiễu muối - tiêu (Salt & Pepper)

**Yêu cầu**: dùng `cv2.medianBlur` để khử nhiễu ảnh `deer_salt.jpg` (median filter đặc biệt hiệu quả với
nhiễu muối - tiêu vì không làm mờ biên ảnh như bộ lọc trung bình).

In [ ]:
noisy = cv2.imread(os.path.join(IMG_DIR, 'deer_salt.jpg'), cv2.IMREAD_GRAYSCALE)

denoised = cv2.medianBlur(noisy, 5)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh bị nhiễu'); plt.imshow(noisy, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau khử nhiễu (median)'); plt.imshow(denoised, cmap='gray')
plt.show()

### 5.2. Kết hợp khử nhiễu và phân ngưỡng

Ảnh `needle1.png`, `needle2.png` chứa các cây kim bị nhiễu. **Yêu cầu**: với mỗi ảnh, kết hợp
`cv2.medianBlur` (khử nhiễu) và phân ngưỡng (`threshold_transform` đã viết ở Mục 1, hoặc dùng
`img < T` / `img > T` trực tiếp) để làm rõ hình dạng cây kim. Hãy tự chọn kernel size và ngưỡng T phù hợp
cho từng ảnh (không nhất thiết phải giống nhau).

In [ ]:
def clean_needle(path, ksize, T, keep_above=True):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    filtered = cv2.medianBlur(img, ksize)
    if keep_above:
        result = np.where(filtered > T, 255, 0).astype(np.uint8)
    else:
        result = np.where(filtered < T, 255, 0).astype(np.uint8)
    return img, result

for name, ksize, T, keep_above in [
    ('needle1.png', 3, 100, False),
    ('needle2.png', 13, 60, True),
]:
    original, result = clean_needle(os.path.join(IMG_DIR, name), ksize, T, keep_above)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.title(f'{name} - gốc'); plt.imshow(original, cmap='gray')
    plt.subplot(1, 2, 2); plt.title(f'{name} - sau xử lý'); plt.imshow(result, cmap='gray')
    plt.show()

## 6. Bài tập tổng hợp: cải thiện ảnh thực tế

Với mỗi ảnh dưới đây, hãy chọn kỹ thuật phù hợp trong số các kỹ thuật đã học ở trên.

### 6.1. Ảnh Mặt trăng (`moon_dark.jpg`) — ảnh có dải cường độ hẹp

**Yêu cầu**: viết hàm `linear_stretch(image)` kéo giãn tuyến tính dải cường độ sáng về [0, 255]:

$$image'[i,j] = \frac{b-a}{d-c} \times (image[i,j] - c) + a$$

với `c, d` là giá trị nhỏ nhất/lớn nhất của ảnh gốc, `a=0, b=255`.

In [ ]:
def linear_stretch(image):
    a, b = 0, 255
    c = float(image.min())
    d = float(image.max())
    if d == c:
        result = np.zeros_like(image, dtype=np.float64)
    else:
        result = (b - a) * (image.astype(np.float64) - c) / (d - c) + a
    result = np.clip(np.rint(result), a, b)
    return np.uint8(result)

moon = cv2.imread(os.path.join(IMG_DIR, 'moon_dark.jpg'), cv2.IMREAD_GRAYSCALE)
moon_stretched = linear_stretch(moon)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(moon, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau kéo giãn tuyến tính'); plt.imshow(moon_stretched, cmap='gray')
plt.show()

### 6.2. Ảnh Bóng bay (`balloon.jpg`) — ảnh tối, dải cường độ tương đối đều

**Yêu cầu**: chọn một kỹ thuật cân bằng histogram phù hợp để tăng cường độ tương phản.

In [ ]:
balloon = cv2.imread(os.path.join(IMG_DIR, 'balloon.jpg'), cv2.IMREAD_GRAYSCALE)

balloon_result = cv2.equalizeHist(balloon)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(balloon, cmap='gray')
plt.subplot(1, 2, 2); plt.title('Sau xử lý'); plt.imshow(balloon_result, cmap='gray')
plt.show()

### 6.3. Ảnh cô gái (`girl1_dark.jpg`) — ảnh màu, tối và thiếu tương phản

**Yêu cầu**: dùng `cv2.convertScaleAbs(img, alpha, beta)` (`alpha` điều chỉnh tương phản, `beta` điều
chỉnh độ sáng) để cải thiện ảnh màu. Hãy tự thử vài giá trị `alpha`, `beta` khác nhau để có kết quả tốt.

In [ ]:
girl = cv2.imread(os.path.join(IMG_DIR, 'girl1_dark.jpg'))

alpha, beta = 1.4, 25
girl_result = cv2.convertScaleAbs(girl, alpha=alpha, beta=beta)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.title('Ảnh gốc'); plt.imshow(cv2.cvtColor(girl, cv2.COLOR_BGR2RGB))
plt.subplot(1, 2, 2); plt.title('Sau xử lý'); plt.imshow(cv2.cvtColor(girl_result, cv2.COLOR_BGR2RGB))
plt.show()